### Week 6, Day 2

Before proceeding with our own MCP Server, let's just look at 2 popular marketplaces for what's out there:

https://glama.ai/mcp  
https://smithery.ai/servers 

We're about to create and use our own MCP Server!

It's pretty simple, but it's not super-simple. The excitement around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

backend/accounts.py

In [ ]:
import os
import subprocess
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters
from IPython.display import display, Markdown

# os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")
load_dotenv(override=True)

True

In [24]:
# ADK passes this stderr target to the MCP stdio client. DEVNULL also avoids a
# Windows Jupyter-kernel fileno error from the server's startup logging.


## Any guesses where this Account python module came from?!

I didn't write it!

In [25]:
from backend.accounts import Account

In [26]:
account = Account.get("Ed")
account.reset()
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [27]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 8967.799719999999, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 344.06676, "timestamp": "2026-08-27 23:07:18", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-08-27 23:07:18", 9997.939719999998]], "total_portfolio_value": 9997.939719999998, "total_profit_loss": -2.0602800000015122}'

In [28]:
account.report()

'{"name": "ed", "balance": 8967.799719999999, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 344.06676, "timestamp": "2026-08-27 23:07:18", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-08-27 23:07:18", 9997.939719999998], ["2026-08-27 23:07:18", 9997.939719999998]], "total_portfolio_value": 9997.939719999998, "total_profit_loss": -2.0602800000015122}'

In [29]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 344.06676,
  'timestamp': '2026-08-27 23:07:18',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [30]:
# Now let's use our accounts server as an MCP server with Google ADK

params = StdioServerParameters(
    command="uv",
    args=["run", "-m", "backend.accounts_server"],
)
account_tools = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=params,
        timeout=30,
    ),
    errlog=subprocess.DEVNULL,
)

In [34]:
account_tools

In [32]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gemini-3.1-flash-lite"

In [33]:
agent = LlmAgent(
    name="account_manager",
    instruction=instructions,
    model=model,
    tools=[account_tools],
)

runner = InMemoryRunner(agent=agent)
events = await runner.run_debug(request, verbose=True)
final_output = next(
    event.content.parts[0].text
    for event in reversed(events)
    if event.is_final_response() and event.content and event.content.parts
)
display(Markdown(final_output))

account_manager > [Calling tool: get_balance({'name': 'Ed'})]
account_manager > [Calling tool: get_holdings({'name': 'Ed'})]
account_manager > [Tool result: {'content': [{'type': 'text', 'text': '8967.799719999999'}], 'structuredContent': {'result': 8967.79...]
account_manager > [Tool result: {'content': [{'type': 'text', 'text': '{\n  "AMZN": 3\n}'}], 'structuredContent': {'AMZN': 3}, 'isEr...]
account_manager > Ed, your account balance is $8,967.80. You currently hold 3 shares of AMZN.


Ed, your account balance is $8,967.80. You currently hold 3 shares of AMZN.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to send a push notification, and then
            enjoy the outcome! There is a solution in backend/push_server.py if you need a hint.
            </span>
        </td>
    </tr>
</table>